# 1. CausalPFN Comparison

This section compares the attribution behavior of CausalPFN against the Bayesian MMM benchmark.

The primary objective is to evaluate whether CausalPFN can recover comparable channel attribution patterns without explicitly specified marketing structure such as adstock and saturation transformations.

In [10]:
# Step 1: Define Bayesian MMM benchmark results

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

channels = [
    "tv_S",
    "ooh_S",
    "print_S",
    "facebook_S",
    "search_S"
]

bayesian_df = pd.DataFrame({
    "channel": channels,

    # Bayesian MMM total channel contribution
    "bayesian_contribution": [
        4.573592,
        1.829350,
        1.668242,
        3.276719,
        2.464963
    ],

    # Bayesian MMM estimated ROAS
    "bayesian_roas": [
        0.260893,
        0.105491,
        0.079605,
        0.126436,
        0.048213
    ]
})

# Add Bayesian direction and ranking
bayesian_df["bayesian_direction"] = np.where(
    bayesian_df["bayesian_contribution"] > 0,
    "Positive",
    "Negative"
)

bayesian_df["bayesian_contribution_rank"] = bayesian_df["bayesian_contribution"].rank(
    ascending=False
)

bayesian_df["bayesian_roas_rank"] = bayesian_df["bayesian_roas"].rank(
    ascending=False
)

bayesian_df

,channel,bayesian_contribution,bayesian_roas,bayesian_direction,bayesian_contribution_rank,bayesian_roas_rank
0,tv_S,4.573592,0.260893,Positive,1.0,1.0
1,ooh_S,1.829350,0.105491,Positive,4.0,3.0
2,print_S,1.668242,0.079605,Positive,5.0,4.0
3,facebook_S,3.276719,0.126436,Positive,2.0,2.0
4,search_S,2.464963,0.048213,Positive,3.0,5.0


In [11]:
# Step 2: Add CausalPFN results using exact mean CATE values from the CausalPFN notebook

comparison_df = bayesian_df.copy()

comparison_df["raw_causal_pfn"] = [
    381131.031250,    # tv_S
    -65830.890625,    # ooh_S
    -46365.781250,    # print_S
    -3943.100098,     # facebook_S
    -158678.734375    # search_S
]

comparison_df["adstock_causal_pfn"] = [
    908742.800000,    # tv_S
    -981255.200000,   # ooh_S
    725554.100000,    # print_S
    -354230.800000,   # facebook_S
    -1145778.000000   # search_S
]

comparison_df

,channel,bayesian_contribution,bayesian_roas,bayesian_direction,bayesian_contribution_rank,bayesian_roas_rank,raw_causal_pfn,adstock_causal_pfn
0,tv_S,4.573592,0.260893,Positive,1.0,1.0,381131.031250,908742.8
1,ooh_S,1.829350,0.105491,Positive,4.0,3.0,-65830.890625,-981255.2
2,print_S,1.668242,0.079605,Positive,5.0,4.0,-46365.781250,725554.1
3,facebook_S,3.276719,0.126436,Positive,2.0,2.0,-3943.100098,-354230.8
4,search_S,2.464963,0.048213,Positive,3.0,5.0,-158678.734375,-1145778.0


In [12]:
# Step 3: Compute sign agreement

comparison_df["raw_causal_sign"] = np.where(
    comparison_df["raw_causal_pfn"] > 0,
    "Positive",
    "Negative"
)

comparison_df["adstock_causal_sign"] = np.where(
    comparison_df["adstock_causal_pfn"] > 0,
    "Positive",
    "Negative"
)

# Agreement rates
raw_sign_agreement = (
    comparison_df["bayesian_direction"]
    ==
    comparison_df["raw_causal_sign"]
).mean()

adstock_sign_agreement = (
    comparison_df["bayesian_direction"]
    ==
    comparison_df["adstock_causal_sign"]
).mean()

print(f"Raw CausalPFN sign agreement: {raw_sign_agreement:.0%}")
print(f"Adstock CausalPFN sign agreement: {adstock_sign_agreement:.0%}")

# Show directions
comparison_df[
    [
        "channel",
        "bayesian_direction",
        "raw_causal_sign",
        "adstock_causal_sign"
    ]
]

Raw CausalPFN sign agreement: 20%
Adstock CausalPFN sign agreement: 40%


,channel,bayesian_direction,raw_causal_sign,adstock_causal_sign
0,tv_S,Positive,Positive,Positive
1,ooh_S,Positive,Negative,Negative
2,print_S,Positive,Negative,Positive
3,facebook_S,Positive,Negative,Negative
4,search_S,Positive,Negative,Negative


## Sign Agreement Analysis

The sign agreement comparison shows that the raw CausalPFN model only correctly recovered the positive attribution direction for TV advertising relative to the Bayesian MMM benchmark, resulting in a sign agreement of 20%.

After introducing adstock preprocessing, the Adstock CausalPFN model correctly recovered the positive attribution direction for both TV and Print advertising, improving the sign agreement to 40%.

These findings suggest that incorporating temporal carryover structure partially improves the ability of CausalPFN to recover marketing attribution behavior. However, the PFN-based approach still fails to fully reproduce the attribution patterns estimated by the Bayesian MMM benchmark.

In [13]:
# Step 4: Display sign agreement summary

sign_agreement_summary = pd.DataFrame({
    "model": ["Raw CausalPFN", "Adstock CausalPFN"],
    "sign_agreement_with_bayesian": [
        raw_sign_agreement,
        adstock_sign_agreement
    ]
})

sign_agreement_summary["sign_agreement_percent"] = (
    sign_agreement_summary["sign_agreement_with_bayesian"] * 100
)

sign_agreement_summary

,model,sign_agreement_with_bayesian,sign_agreement_percent
0,Raw CausalPFN,0.2,20.0
1,Adstock CausalPFN,0.4,40.0


## Sign Agreement Findings

The sign agreement analysis reveals substantial differences between the Bayesian MMM benchmark and the attribution patterns recovered by CausalPFN.

The raw CausalPFN model achieved only 20% sign agreement with the Bayesian MMM benchmark, correctly identifying the attribution direction only for TV advertising.

After introducing adstock preprocessing, the agreement increased to 40%, with the model correctly recovering the attribution direction for both TV and Print advertising.

These results suggest that incorporating temporal carryover structure partially improves the ability of PFN-based models to recover marketing attribution behavior. However, the relatively low agreement rates also indicate that implicit learned priors alone may not be sufficient to fully reproduce the attribution structure estimated by a Bayesian MMM with explicitly specified adstock and saturation assumptions.

In [14]:
# Step 5: Compare channel rankings

comparison_df["raw_causal_rank"] = comparison_df["raw_causal_pfn"].rank(
    ascending=False
)

comparison_df["adstock_causal_rank"] = comparison_df["adstock_causal_pfn"].rank(
    ascending=False
)

ranking_comparison = comparison_df[
    [
        "channel",
        "bayesian_contribution_rank",
        "raw_causal_rank",
        "adstock_causal_rank"
    ]
]

ranking_comparison

,channel,bayesian_contribution_rank,raw_causal_rank,adstock_causal_rank
0,tv_S,1.0,1.0,1.0
1,ooh_S,4.0,4.0,4.0
2,print_S,5.0,3.0,2.0
3,facebook_S,2.0,2.0,3.0
4,search_S,3.0,5.0,5.0


## Ranking Comparison Findings

The ranking comparison reveals partial alignment between the Bayesian MMM benchmark and the attribution patterns recovered by CausalPFN.

TV advertising remained the highest-ranked channel across all models, indicating that both the Bayesian MMM and the PFN-based approaches consistently identify TV as the dominant marketing channel.

OOH advertising also exhibited relatively stable low rankings across models. In contrast, substantial differences emerged for Search, Print, and Facebook advertising.

The raw CausalPFN model produced rankings that only partially matched the Bayesian benchmark, while the Adstock CausalPFN model shifted the ranking structure further by strongly increasing the relative importance of Print advertising.

Overall, the results suggest that PFN-based models are capable of recovering certain broad attribution patterns, particularly for dominant channels such as TV. However, the models still struggle to fully reproduce the channel importance ordering estimated by the Bayesian MMM benchmark.

# 2. TabPFN Comparison

This section compares the predictive and attribution behavior of TabPFN against the Bayesian MMM benchmark.

Unlike the Bayesian MMM, TabPFN does not explicitly encode marketing-specific structures such as adstock and saturation. Therefore, the comparison evaluates whether TabPFN can recover comparable forecasting performance and channel-level importance patterns using its implicit learned prior.

In [15]:
# Step 1: Compare forecasting performance between Bayesian MMM and TabPFN models

tabpfn_forecast_df = pd.DataFrame({
    "model": [
        "Bayesian MMM",
        "TabPFN",
        "TabPFN + Adstock",
        "TabPFN + Adstock + Saturation"
    ],
    "mape": [
        8.53,
        7.97,
        7.41,
        7.68
    ]
})

tabpfn_forecast_df

,model,mape
0,Bayesian MMM,8.53
1,TabPFN,7.97
2,TabPFN + Adstock,7.41
3,TabPFN + Adstock + Saturation,7.68


## Forecasting Performance Findings

The forecasting comparison shows that all TabPFN-based models achieved lower MAPE values than the Bayesian MMM benchmark.

| Model | MAPE |
|---|---|
| Bayesian MMM | 8.53% |
| TabPFN | 7.97% |
| TabPFN + Adstock | 7.41% |
| TabPFN + Adstock + Saturation | 7.68% |

The plain TabPFN model already outperformed the Bayesian MMM in predictive accuracy. Introducing adstock preprocessing further improved forecasting performance and produced the lowest overall MAPE value.

However, adding saturation preprocessing slightly reduced forecasting performance relative to the adstock-only TabPFN model.

These findings suggest that TabPFN is highly effective at capturing predictive temporal patterns in the marketing dataset, particularly when temporal carryover structure is introduced through adstock preprocessing.

In [16]:
# Step 2: Compare Bayesian MMM vs TabPFN channel rankings

tabpfn_ranking_df = pd.DataFrame({
    "channel": [
        "tv_S",
        "ooh_S",
        "print_S",
        "facebook_S",
        "search_S"
    ],

    # Bayesian MMM ranking
    "bayesian_rank": [
        1,
        4,
        5,
        2,
        3
    ],

    # TabPFN ranking
    "tabpfn_rank": [
        1,
        4,
        2,
        3,
        5
    ],

    # TabPFN + Adstock ranking
    "tabpfn_adstock_rank": [
        1,
        4,
        2,
        3,
        5
    ],

    # TabPFN + Adstock + Saturation ranking
    "tabpfn_adstock_saturation_rank": [
        1,
        4,
        2,
        3,
        5
    ]
})

tabpfn_ranking_df

,channel,bayesian_rank,tabpfn_rank,tabpfn_adstock_rank,tabpfn_adstock_saturation_rank
0,tv_S,1,1,1,1
1,ooh_S,4,4,4,4
2,print_S,5,2,2,2
3,facebook_S,2,3,3,3
4,search_S,3,5,5,5


## Attribution Ranking Findings

The ranking comparison reveals partial alignment between the Bayesian MMM benchmark and the TabPFN-based models.

Across all models, TV advertising consistently remained the highest-ranked channel, indicating that both the Bayesian MMM and TabPFN approaches strongly identify TV as the dominant marketing driver.

Similarly, OOH advertising consistently remained among the weakest channels across all specifications.

However, important differences emerged for Print, Facebook, and Search advertising. While the Bayesian MMM ranked Facebook and Search above Print, all TabPFN-based models strongly increased the relative importance of Print advertising and reduced the ranking of Search.

Notably, introducing adstock and saturation preprocessing did not materially change the TabPFN attribution ordering. This suggests that the predictive improvements observed earlier do not necessarily translate into attribution patterns that fully align with the Bayesian MMM benchmark.

Overall, the results indicate that TabPFN is capable of recovering certain broad channel importance patterns, particularly for dominant channels such as TV. However, the model still struggles to fully reproduce the detailed attribution structure estimated by the Bayesian MMM.